# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [1]:
#Libraries
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor,AdaBoostRegressor, GradientBoostingRegressor

from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.metrics import classification_report

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [3]:

X = spaceship.drop(columns='Transported')
y = spaceship['Transported']

print(f"Datensatzgröße: {X.shape}")
print(f"Klassen: {spaceship}")
print(f"Klassenverteilung:")
print(f"  Not Transported (0): {sum(y==False)}")
print(f"  Transported  (1): {sum(y==True)}")
print()
X.head()

Datensatzgröße: (8693, 13)
Klassen:      PassengerId HomePlanet CryoSleep     Cabin    Destination   Age    VIP  \
0        0001_01     Europa     False     B/0/P    TRAPPIST-1e  39.0  False   
1        0002_01      Earth     False     F/0/S    TRAPPIST-1e  24.0  False   
2        0003_01     Europa     False     A/0/S    TRAPPIST-1e  58.0   True   
3        0003_02     Europa     False     A/0/S    TRAPPIST-1e  33.0  False   
4        0004_01      Earth     False     F/1/S    TRAPPIST-1e  16.0  False   
...          ...        ...       ...       ...            ...   ...    ...   
8688     9276_01     Europa     False    A/98/P    55 Cancri e  41.0   True   
8689     9278_01      Earth      True  G/1499/S  PSO J318.5-22  18.0  False   
8690     9279_01      Earth     False  G/1500/S    TRAPPIST-1e  26.0  False   
8691     9280_01     Europa     False   E/608/S    55 Cancri e  32.0  False   
8692     9280_02     Europa     False   E/608/S    TRAPPIST-1e  44.0  False   

      RoomServi

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines


In [4]:
# Handle Missing Values in Categorical Columns
cat_cols = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP'] 
for col in cat_cols:
    X[col] = X[col].fillna(X[col].mode()[0])

/var/folders/gn/zvx5kxqj5ng9ng0cg5f15yk80000gn/T/ipykernel_24564/1511286050.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X[col] = X[col].fillna(X[col].mode()[0])


In [5]:
# Handle Missing Values in Numerical Columns
num_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
for col in num_cols:
    X[col] = X[col].fillna(X[col].mode()[0])

In [6]:
# Drop 
X = X.drop(columns=['PassengerId', 'Cabin', 'Name'])

# Encode
X = pd.get_dummies(X, columns=['HomePlanet', 'Destination', 'CryoSleep', 'VIP'])

In [7]:
y = spaceship['Transported'].astype(int)

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)
print(f"Training: {X_train.shape[0]} | Test: {X_test.shape[0]}")

Training: 6519 | Test: 2174


In [9]:
# Feature Scaling 
# Normalisieren
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training: {X_train.shape} | Test: {X_test.shape}")

Training: (6519, 16) | Test: (2174, 16)


- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

In [10]:
from sklearn.ensemble import RandomForestClassifier

# Random Forest: Bagging + random feature selection at each split
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print(f"Random Forest Accuracy: {rf.score(X_test, y_test):.4f}")
print(classification_report(y_test, y_pred_rf))

Random Forest Accuracy: 0.7838
              precision    recall  f1-score   support

           0       0.79      0.77      0.78      1077
           1       0.78      0.80      0.79      1097

    accuracy                           0.78      2174
   macro avg       0.78      0.78      0.78      2174
weighted avg       0.78      0.78      0.78      2174



- Evaluate your model

In [11]:
# Baseline evaluation — Random Forest with default hyperparameters
print(f"Baseline Accuracy: {rf.score(X_test, y_test):.4f}")
print(classification_report(y_test, y_pred_rf))

Baseline Accuracy: 0.7838
              precision    recall  f1-score   support

           0       0.79      0.77      0.78      1077
           1       0.78      0.80      0.79      1097

    accuracy                           0.78      2174
   macro avg       0.78      0.78      0.78      2174
weighted avg       0.78      0.78      0.78      2174



**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

In [12]:
# Correct grid for RandomForestClassifier
# Note: learning_rate does NOT exist in RandomForest — that's GradientBoosting!
grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 20],
    'min_samples_split': [2, 5, 10]
}
# 3 x 3 x 3 = 27 combinations, cv=3 → 81 total fits

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=grid,
    cv=3,
    scoring='accuracy',   # classification → accuracy, not r2!
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)  # no scaling needed for Random Forest

print("Best Parameters:", grid_search.best_params_)
print(f"Best CV Accuracy: {grid_search.best_score_:.4f}")

Fitting 3 folds for each of 27 candidates, totalling 81 fits
Best Parameters: {'max_depth': 10, 'min_samples_split': 10, 'n_estimators': 200}
Best CV Accuracy: 0.7938


- Run Grid Search

- Evaluate your model

In [ ]:
# Evaluate tuned model vs baseline
best_model = grid_search.best_estimator_
y_pred_tuned = best_model.predict(X_test)

print(f"Baseline Accuracy: {rf.score(X_test, y_test):.4f}")
print(f"Tuned Accuracy:    {best_model.score(X_test, y_test):.4f}")
print(f"Improvement:       +{(best_model.score(X_test, y_test) - rf.score(X_test, y_test)):.4f}")
print()
print(classification_report(y_test, y_pred_tuned))

Baseline Accuracy: 0.7838
Tuned Accuracy:    0.7953
Improvement:       +0.0115

              precision    recall  f1-score   support

           0       0.81      0.77      0.79      1077
           1       0.78      0.82      0.80      1097

    accuracy                           0.80      2174
   macro avg       0.80      0.80      0.80      2174
weighted avg       0.80      0.80      0.80      2174



Exception ignored in: <function ResourceTracker.__del__ at 0x1089d9bc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x1064adbc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x1078d9bc0>
Traceback (most recent call last